# 01b — Train SDXL Character LoRA (ai-toolkit)

Trains an SDXL character LoRA from your captioned reference images.

**Runtime:** A100 (40 GB) recommended. Training 2000 steps with rank 16 takes ~20–40 min on A100.
T4 (16 GB) works for smaller rank / lower batch but is tight and slower.

**Prerequisites:** Run `01a_caption_refs.ipynb` first to prepare images + captions.

**Output:** `<DRIVE_BASE>/loras/<CHARACTER_NAME>_sdxl.safetensors`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'MarcD'
TRIGGER_TOKEN  = 'sks_marcd'
TRAIN_STEPS    = 2250    # 1500–3000; bump to 2500 if identity is weak
LORA_RANK      = 32      # 32 for more identity detail (needs ~16 GB VRAM)
LEARNING_RATE  = '1.0e-4'
# ─────────────────────────────────────────────────────────────────────────

DRIVE_BASE  = '/content/drive/MyDrive/ai_character_studio'
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
LORAS_DIR   = f'{DRIVE_BASE}/loras'
OUTPUT_LORA = f'{LORAS_DIR}/{CHARACTER_NAME}_sdxl.safetensors'

import os
os.makedirs(LORAS_DIR, exist_ok=True)

refs = [f for f in os.listdir(REF_DIR) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
print(f'Character: {CHARACTER_NAME} | Trigger: {TRIGGER_TOKEN}')
print(f'Reference images: {len(refs)}')
if len(refs) < 10:
    print('WARNING: fewer than 10 images may cause weak identity learning.')

In [ ]:
# Switching to diffusers official DreamBooth LoRA training script.
# This uses the system diffusers/transformers (no version conflicts).
# The official script is well-maintained and produces identical LoRA results to ai-toolkit.

# Get the official diffusers training examples
EXAMPLES_DIR = '/content/diffusers_examples'
!git clone --depth 1 https://github.com/huggingface/diffusers.git {EXAMPLES_DIR} 2>/dev/null || \
    git -C {EXAMPLES_DIR} pull --ff-only 2>/dev/null

TRAIN_SCRIPT = f'{EXAMPLES_DIR}/examples/dreambooth/train_dreambooth_lora_sdxl.py'

# Install training script dependencies
!pip install -q accelerate bitsandbytes prodigyopt peft datasets
!pip install -q "peft>=0.6.0"

# Verify
import os
assert os.path.exists(TRAIN_SCRIPT), f"Training script not found: {TRAIN_SCRIPT}"
print(f'Training script ready: {TRAIN_SCRIPT}')

import transformers, diffusers
print(f'  transformers: {transformers.__version__}')
print(f'  diffusers: {diffusers.__version__}')

In [ ]:
# Check GPU
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'CUDA: {torch.cuda.is_available()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
import os, json

# Accelerate config — use bf16 on A100
ACCEL_CONFIG = '/content/accelerate_config.yaml'
with open(ACCEL_CONFIG, 'w') as f:
    f.write("""compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
mixed_precision: 'bf16'
num_machines: 1
num_processes: 1
""")

# Validation prompt — used to check identity at each sample interval
VALIDATION_PROMPT = f'{TRIGGER_TOKEN}, portrait photo, detailed face, dramatic lighting'

# How many epochs — diffusers script trains by epochs, not raw steps.
# With 11 images, batch=1: 1 epoch = 11 steps. For ~2250 steps → ~205 epochs.
N_IMAGES = len([f for f in os.listdir(REF_DIR) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))])
N_EPOCHS = max(1, TRAIN_STEPS // max(1, N_IMAGES))

print(f'Training config:')
print(f'  Character: {CHARACTER_NAME} | Trigger: {TRIGGER_TOKEN}')
print(f'  Images: {N_IMAGES} | Steps target: {TRAIN_STEPS}')
print(f'  Epochs: {N_EPOCHS} ({N_EPOCHS * N_IMAGES} effective steps)')
print(f'  Rank: {LORA_RANK} | LR: {LEARNING_RATE}')
print(f'  Validation prompt: {VALIDATION_PROMPT}')

In [ ]:
import time, os

TRAIN_OUTPUT_DIR = '/content/training_output'
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)

start = time.time()
!ACCELERATE_CONFIG_FILE={ACCEL_CONFIG} accelerate launch \
    --config_file={ACCEL_CONFIG} \
    {TRAIN_SCRIPT} \
    --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
    --instance_data_dir="{REF_DIR}" \
    --instance_prompt="{TRIGGER_TOKEN}" \
    --output_dir="{TRAIN_OUTPUT_DIR}" \
    --mixed_precision="bf16" \
    --num_train_epochs={N_EPOCHS} \
    --train_batch_size=1 \
    --gradient_accumulation_steps=1 \
    --learning_rate={LEARNING_RATE} \
    --lr_scheduler="constant" \
    --lr_warmup_steps=0 \
    --rank={LORA_RANK} \
    --lora_alpha={LORA_RANK // 2} \
    --validation_prompt="{VALIDATION_PROMPT}" \
    --validation_epochs=25 \
    --seed=42 \
    --checkpointing_steps=500 \
    --gradient_checkpointing

elapsed = time.time() - start
print(f'\nTraining finished in {elapsed/60:.1f} minutes.')

In [ ]:
# Copy the final LoRA to Drive
import glob, shutil

# ai-toolkit saves LoRAs to training_folder/name/
candidates = glob.glob(f'/content/training_output/{CHARACTER_NAME}_sdxl/*.safetensors')
# Pick the one with highest step number (the final checkpoint)
if not candidates:
    # Try without step suffix (some versions save as <name>.safetensors)
    candidates = glob.glob(f'/content/training_output/**/*.safetensors', recursive=True)

if candidates:
    # Sort by modification time — latest is the final
    latest = max(candidates, key=os.path.getmtime)
    shutil.copy2(latest, OUTPUT_LORA)
    print(f'✅ LoRA saved to Drive: {OUTPUT_LORA}')
    size_mb = os.path.getsize(OUTPUT_LORA) / 1024**2
    print(f'   Size: {size_mb:.1f} MB')
else:
    print('ERROR: No .safetensors found in training output. Check the training log above.')

In [ ]:
# Update metadata.json on Drive
import json
meta_path = f'{CHAR_DIR}/metadata.json'
metadata = {}
if os.path.exists(meta_path):
    with open(meta_path) as f:
        metadata = json.load(f)

metadata.update({
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'sdxl',
    'lora_path': OUTPUT_LORA,
    'train_steps': TRAIN_STEPS,
    'lora_rank': LORA_RANK,
})
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'metadata.json updated: {meta_path}')
print(json.dumps(metadata, indent=2))

In [ ]:
# Quick validation — generate 2 sample images using diffusers pipeline
# (This is just a fast check; full validation is done via ComfyUI in 02_test_stills.ipynb)
from diffusers import DiffusionPipeline, AutoencoderKL
import torch
from PIL import Image

vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    vae=vae, torch_dtype=torch.float16, variant='fp16'
).to('cuda')
pipe.load_lora_weights(OUTPUT_LORA)

prompts = [
    f'{TRIGGER_TOKEN}, portrait, detailed face, dramatic lighting',
    f'{TRIGGER_TOKEN}, full body, standing, outdoor scene',
]
images = pipe(prompts, num_inference_steps=30, guidance_scale=7.0).images

sample_dir = f'{CHAR_DIR}/samples'
os.makedirs(sample_dir, exist_ok=True)
for i, img in enumerate(images):
    path = f'{sample_dir}/validation_{i:02d}.png'
    img.save(path)
    print(f'Sample saved: {path}')

# Display
from IPython.display import display
for img in images:
    display(img.resize((512, 512)))

print('\n✅ Training complete. Check the samples above vs your reference images.')
print('If face is drifting: bump TRAIN_STEPS to 2500 or LORA_RANK to 32 and retrain.')
print('Next: run 02_test_stills.ipynb for full ComfyUI stills generation.')